# Task 4: graph–text retrieval

Frozen encoder outputs → normalized projection heads → symmetric InfoNCE. Synthetic mode is an offline pipeline check, not music retrieval evidence.

In [ ]:
from pathlib import Path
import sys, json
from datetime import datetime
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.task3.data import make_demo
from src.task4.train import train, evaluate
from IPython.display import display, Image
import pandas as pd

In [ ]:
USE_SYNTHETIC = True
FEATURES = ROOT / 'data/processed/task3/features.pt'
OUTPUT = ROOT / ('tmp/task4_notebook' if USE_SYNTHETIC else 'results/task4') / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
if USE_SYNTHETIC:
    FEATURES = make_demo(OUTPUT.with_suffix('.demo.pt'))
print('Synthetic diagnostics' if USE_SYNTHETIC else 'Real paired features')

In [ ]:
report = train(FEATURES, OUTPUT, epochs=3 if USE_SYNTHETIC else 30, device='auto')
print(json.dumps(report, indent=2))
display(Image(filename=str(OUTPUT / 'learning_curves.png')))
display(pd.DataFrame(json.loads((OUTPUT / 'validation/examples.json').read_text())))

## Optional final test

Enable after freezing settings. The full split is the retrieval gallery. R@K uses min(K, gallery size); R@10 is trivial for the six-pair demo gallery. To evaluate an existing run, set OUTPUT and FEATURES to its paths and run this cell without retraining.

In [ ]:
RUN_TEST = False
if RUN_TEST:
    test_report = evaluate(OUTPUT / 'best_model.pt', FEATURES, OUTPUT / 'test', device='auto')
    print(json.dumps(test_report, indent=2))

## Interpretation

Inspect both retrieval directions and failure queries. Real-data results, CLAP comparison, and human listening evaluation remain outstanding. Attention or embedding similarity does not establish a causal explanation. See `docs/task4/README.md` for terminal commands and assumptions.